In [1]:
!pip install -q transformers[torch]
!pip install -q evaluate
!pip install -q peft
!pip install -q accelerate
!pip install -q trl
!pip install -q flash-attn
!pip install -q bitsandbytes

In [2]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HfArgumentParser,
    Trainer,
    BitsAndBytesConfig,
    pipeline,
    logging
)
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import evaluate
import torch
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
import math
from datasets import DatasetDict, Dataset
from pprint import pprint
from collections import defaultdict
import json
from pandas import json_normalize
from trl import SFTTrainer, setup_chat_format
from google.colab import files
import os
import wandb
import zipfile
import bitsandbytes as bnb


In [3]:
# These are specifically set to match the capability of Colab A100's
print(torch.cuda.get_device_capability()[0]) # if this is above 8, we set to flash attn and bfloat16 instead of float16
attn_implementation = "flash_attention_2"
torch_dtype = torch.bfloat16

8


In [4]:
# Model definition
model_name = "meta-llama/Llama-3.2-3B-Instruct"
lr = 1e-5
batch_size = 4
num_epochs = 3

In [5]:
# setting initial lora hyperparameters, we are going to need to do a random search

"""
setting initial lora hyperparameters, we are going to need to do a random search
r = 4, 8, 16
lora_alpha = 32, 16, 8
lora_dropout = 0.1, 0.2, 0.3

baseline: 16, 32, 0.1
high regularization: 8, 16, 0.2
complex tasks: 32, 64, 0.0
low resource: 4, 8, 0.1
moderately complex tasks: 16, 48, 0.05
"""


baseline_r = 16
baseline_lora_alpha = 32
baseline_lora_dropout = 0.1

grid_r = [8, 32, 4, 16]
grid_lora_alpha = [16, 64, 8, 48]
grid_lora_dropout = [0.2, 0.0, 0.1, 0.05]

# Data Wrangling

In [5]:
# load the data
labeled_data_path = "/content/labeled_data5k.csv"
full_df = pd.read_csv(labeled_data_path)
print(full_df.shape)

(5000, 17)


In [6]:
labeled_df = full_df[["description", "reformatted", "label"]]
labeled_df.head()

,description,reformatted,label
0,"Dear Applicant,Hope you are doing well!!!We ha...",```markdown\n# Job Title: Senior MES Engineer\...,"{""required"":{""education"":[{""field_of_study"":[""..."
1,LIVING OUR VALUES All associates at The Fried...,```markdown\n# Project Manager - Data & Analyt...,"{""required"":{""education"":[{""field_of_study"":[""..."
2,Job Title: AWS Cloud Champion Responsib...,# AWS Cloud Champion\n\n## Responsibilities\n\...,"{""required"":{""education"":[],""experience"":[],""c..."
3,Outstanding long-term contract opportunity! A ...,```markdown\n# Job Title: Data Management Anal...,"{""required"":{""education"":[{""field_of_study"":[]..."
4,We have a 6 month contract to hire position op...,```markdown\n# Network Engineer\n\n## Job Resp...,"{""required"":{""education"":[{""field_of_study"":[""..."


In [7]:
# metric = evaluate.load('accuracy')

# def compute_metrics(p):
#     predictions, labels = p
#     predictions = np.argmax(predictions, axis=1)
#     return metric.compute(predictions=predictions, references=labels)

In [8]:
# Check to make sure that the label came through as correctly formatted JSON
def is_valid_json(s):
    try:
        json.loads(s)
        return True
    except json.JSONDecodeError:
        return False

# Apply the function to identify invalid JSON entries
invalid_json_mask = ~labeled_df["label"].apply(is_valid_json)
invalid_json_rows = labeled_df[invalid_json_mask]

print("Number of invalid JSON entries:", invalid_json_rows.shape[0])
print("Invalid JSON entries:")
print(invalid_json_rows['description'])

Number of invalid JSON entries: 1
Invalid JSON entries:
2507    Job Description :Minimum 10 years of experienc...
Name: description, dtype: object


In [9]:
# Drop invalid json label rows
labeled_df = labeled_df[~invalid_json_mask]

In [10]:
reformat_prompt = """
You are a tool used to organize and reformat job descriptions into a structured markdown format
that is optimized for information extraction from the ChatGPT 4o model. You must retain all
information from the job description that is related to job requirements, job qualifications,
job responsibilities, or job duties. You must exclude any information related to descriptions
of the company, team, salary, travel, location, culture, application process,
physical requirements/demands, or benefits. After, output the
reformatted job description in markdown format.
"""

In [11]:
extract_prompt = """
You are a tool used to extract the requirements and responsibilities from a job description. I will be inputting a series of job descriptions, so please use the following prompt:

First, extract the technical skill requirements and responsibilities from the job description as JSON by following these steps:
Step 1 - Technical skills should be defined as: Specific abilities that relate to the use of tools, technologies, programming languages, software applications, or technical processes.
Step 2 - Identify the technical skills that are explicitly stated in the job description. Do not include soft skills or job titles. Include any technical skills listed as a experience requirement.
Step 3 - Store each of the technical skills in a key called 'skill_name' under an object called “Technical_skills”.
Step 4 - If multiple technical skills in a phrase are grouped using parentheses or listed using grouping connectors similar to "including", "like", "/", "or", or "such as" then the skills should be listed in one array so that they result in a single skill_name. If technical skills are listed without grouping connectors and in a phrase ending in "and" then treat each listed skill as a separate skill_name. When no clear grouping is implied, list each skill individually rather than in an array.  Below are examples of desired outputs:
Example phrase 1 -  "experience of 6+ years using cybersecurity architectures, IDS/IPS, SIEM tools, or firewalls"
Example output 1 - skill_name: [cybersecurity architectures, IDS/IPS, SIEM tools, firewalls]
Example phrase 2 - "knowledge of database systems including MySQL, PostgreSQL, and Redis"
Example output 2 - skill_name: [database systems, MySQL, PostgreSQL, Redis]
Example phrase 3  - "2+ years experience with Java/JEE, SOAP/REST/Micro Services, XML"
Example output 3 (3 separate skill_name keys) - skill_name: [Java, JEE], skill_name: [SOAP, REST, Micro Services], skill_name: [XML]
Example phrase 4  - "experience with SQL Server, Oracle, MongoDB"
Example output 4 (3 separate skill_name keys) - skill_name: [SQL Server], skill_name: [Oracle], skill_name: [MongoDB]
Example phrase 5 -  "4 or more years experience with messaging architectures (EAI), SAP XI, BizTalk, and Web Services"
Example output 5 (4 separate skill_name keys) - skill_name: [messaging architectures, EAI], skill_name: [SAP XI], skill_name: [BizTalk], skill_name: [Web Services]
Example phrase 6 - "AWS Knowledge: Event Bridge, Cloud Watch, Cloud Trail"
Example output 6 (4 separate skill_name keys) - skill_name: [AWS], skill_name: [Event Bridge], skill_name: [Cloud Watch], skill_name: [Cloud Trail],
Example phrase 7 - "Working knowledge of EDB, Oracle, and AWS operations (S3, EC2)."
Example output 7 (3 separate skill_name keys) - skill_name: [EDB], skill_name: [Oracle], skill_name: [AWS Operations, S3, EC2]
Example phrase 8 - "expertise in the SAP Analytics Roadmap and new technologies such as SAP HANA, and SAP SAC."
Example output 8 (2 separate skill_name keys) - skill_name: [SAP Analytics Roadmap], skill_name: [SAP HANA, SAP SAC]
Example phrase 9 - "Demonstrated understanding of WAN, LAN, and VPN networks, including the relationship between network infrastructure and IP-enabled devices."
Example output 9 (4 separate skill_name keys) - skill_name: [WAN], skill_name: [LAN], skill_name: [VPN networks], skill_name: [network infrastructure, IP-enabled devices]
Step 6 - Classify each of the technical skills that are described as mandatory, essential, needed, or similar as ‘Required’. Classify the technical skills that are described as preferred, nice to have, or similar as ‘Preferred’. If a requirement is not mentioned or not clear, then the skill should be classified as ‘Preferred’. If a technical skill is only described under job responsibilities or job duties it should be classified as 'Preferred'. Separate the skills under two objects called ‘Required’ and ‘Preferred’.
Step 7 - For each skill, extract the minimum years required explicitly stated for that specific skill (not inferred or through association) and list 0 if years are not explicitly stated for that specific skill. If a range of years is given, use the smallest year. Label these using a key called ‘minyears’.
Step 8 - If any two technical skills you have extracted are extremely similar, merge them into a single skill_name array.

Next, extract the required level of education from the job description as JSON by following these steps:
Step 1 - Locate the requirements, qualifications, prerequisite, or a similar section of the job description and identify any educational requirements that are explicitly stated.
Step 2 - Create a key called ‘education_level’ that lists the levels of education mentioned in the job qualifications. Categorize the levels of education under one of the following values that fits best: High School Diploma, Associate's, Current Bachelor's Student, Bachelor’s, Current Master's Student, Master’s, Doctorate, Postdoctorate, Vocational, None.
Step 3 - Classify the levels of education that are explicitly described as mandatory, essential, required, or similar as ‘Required’. Classify the levels of education that are explicitly described as preferred, nice to have, or similar as ‘Preferred’.  If a qualification is not mentioned, then the level of education should be classified as ‘Preferred’.  Separate these levels of education under two objects called ‘Required’ and ‘Preferred’.
Step 4 - If more than one level of education meets the necessary requirement, the higher levels of education should be classified as 'Preferred'. If multiple levels of education are preferred, they should all be stored as one level of education using an array of comma separated values. As a result, there should only be one 'education_level' key in each of the 'Required' and 'Preferred' object. For example, phrases like "A Bachelor's is required, a Master's/PhD is preferred" would have Bachelor's under the 'Required' object but [Master's, Doctorate] under the 'Preferred' object.
Step 5 - If experience is an accepted alternative to the education requirement, add 'Or Experience' as a value into the 'education_level' array but only if there is an existing education level. Below are examples of desired outputs:
Example phrase 1 - "a Master's degree in mathematics, STEM, or a related field, or 2 years of experience"
Example output 1 - education_level: [Master's, Or Experience]
Example phrase 2- " graduate in information systems or equivalent hands-on work experience"
Example output 2 - education_level: [Bachelor's, Or Experience]
Example phrase 3 - "**Option A**: Bachelor's or higher degree with at least 30 semester hours in mathematics and physical sciences.- **Option B**: Combination of education and experience equivalent to a 4-year course of study."
Example output 3- education_level: [Current Bachelor's, Or Experience]
Step 6 - For each level of education in the list, identify and include any fields of study mentioned in a key called ‘field_of_study’ as separate values in an array. If terms like  "or similar", "or equivalent", or "or related" are used, include ‘Related’ as a field of study. If no fields of study are listed then do not list anything. Below are examples of desired outputs:
Example phrase 1 - "A Bachelor's degree in Computer Science, STEM, or a related field"
Example output 1 - field_of_study: [Computer Science, STEM, Related]
Example phrase 2 - "A graduate in Computer Science, Business, STEM, or equivalent"
Example output 2 - field_of_study: [Computer Science, STEM, Related]

Next extract any required credentials, certifications, licenses, or similar credentials from the job description as JSON by following these steps:
Step 1 - Search within the job description for certifications, licenses, security clearances, or similar credential qualifications that are related to the job.
Step 2 - Extract the names of any certifications, licenses, security clearances, or similar credentials that are explicitly stated under keys called ‘credential’. Do not include credential qualifications like driver's licenses, citizenship documentation, or other credentials that are not related to the job.
Step 3 - If multiple options are acceptable, place them in an array under the credential. If both a credential and it's abbreviation/acronym are given, provide both in the array. Below is an example of a desired output:
Example phrase 1 - "Certified as IAT or IAM Level III"
Example output 1- field_of_study:  [IAM Level III Certification, IAT Level III Certification]
Example phrase 2 - "CCNP (Cisco Certified Network Professional)"
Example output 2 - field_of_study:  [CCNP, Cisco Certified Network Professional]
Step 4 - Classify each of these credentials that are explicitly described as mandatory, essential, required, or similar as ‘Required’. Classify the credentials that are explicitly described as preferred, nice to have, or similar as ‘Preferred’. If a qualification is not mentioned, then the credential should be classified as ‘Preferred’. Separate these credentials under two objects called ‘Required’ and ‘Preferred’.
Step 5 - If any certifications, licenses, security clearances, or similar credentials are listed as a technical skill, remove them from the technical skill section.
Step 6 - The output should be structured under the key ‘Credentials’, under objects for ‘Required’ and ‘Preferred’.

Next, extract the experience requirements and responsibilities from the job description as JSON by following these steps:
Step 1 - Experience and responsibilities are defined as: The amount and type of prior work experience a candidate must have in order to qualify for the job or to complete the duties of the job. This includes the level of seniority and industry background.
Step 2 - Locate and identify any experience requirements and responsibilities that are explicitly stated in the job description.
Step 3 - If any experience requirements involve technical skills, they should be not be included.
Step 4 - Create a key for each experience and responsibility under a key called 'experience_desc'.
Step 5 - When extracting experiences and responsibilities, if multiple experiences/responsibilities are listed in a phrase using connectors similar to "including", "like", "/", "or", or "such as"  then they should each be listed in one array so they result in a single experience_desc. However, If multiple experiences/responsibilities are listed in a phrase ending in "and" then list each as a separate experience_desc. Below are examples of desired outputs:
Example phrase 1 - "7+ years overall experience in product management, marketing analysis, or professional services"
Example output 1 - experience_desc: [product management, marketing analysis, professional services]
Example phrase 2 - "must have leadership experience (manager, director, or VP level)"
Example output 2 - experience_desc: [manager, director, VP]
Example phrase 3 - "3+ years in designing, implementing, and optimizing custom solutions for enterprise software applications"
Example output 3 (3 separate experience_desc keys) - experience_desc: [designing custom solutions for enterprise software applications], experience_desc: [implementing custom solutions for enterprise software applications], experience_desc: [optimizing custom solutions for enterprise software applications]
Step 6 - If terms like "or similar", "or equivalent", or "or related" are used, include ‘Related’ in the array.  Below are examples of desired outputs:
Example phrase 1 - "experience as a data scientist or a related role"
Example output 1 - experience_desc: [data scientist, Related]
Example phrase 2 - "5+ years of data engineering experience or equivalent experience"
Example output 2 - experience_desc: [data engineering, Related]
Step 7 - If experience is listed as a requirement without specifying a specific type of experience, then list the experience as 'Work Experience'. Below are examples of desired outputs:
Example phrase 1 - "4+ years of experience are required"
Example output 1 - experience_desc: [Work Experience]
Example phrase 2 - "Experience Required: 5 years"
Example output 2 - experience_desc: [Work Experience]
Step 8 - Identify the educational requirements in the job description. If the educational requirements allow work experience as an alternative to education, using phrases like "equivalent experience," "equivalent hands-on work experience," or "professional experience," you must add a experience_desc with the value ['Work Experience', 'Or Education']. This rule applies to any phrasing where work experience is presented as an acceptable substitute for education. Below are examples of desired outputs:
Example phrase 1 - "Required education: A degree in engineering or 3+ years of relevant industry experience"
Example output 1 - experience_desc: [Work Experience, Or Education]
Example phrase 2 - "A Bachelor's degree in computer science (or equivalent experience) is needed"
Example output 2 - experience_desc: [Work Experience, Or Education]
Example phrase 3 - "Master of Science degree in computer science, MIS, or equivalent hands-on work experience"
Example output 3 - experience_desc: [Work Experience, Or Education]
Example phrase 4 - "A degree from an accredited university but equivalent professional experience is also acceptable"
Example output 4 - experience_desc: [Work Experience, Or Education]
Step 9 - For each experience_desc in the list, identify and include any industries mentioned in a key called ‘industry’ as separate values in an array. If terms like  "or similar", "or equivalent", or "or related" are used, include ‘Related’ as an industry.  If no industries are listed then do not list anything. Below are examples of desired outputs:
Example phrase 1 - "Minimum of 5+ years of experience as a Project Manager / Scrum Master in the AI/ML industry"
Example output 1 - industry: [AI Industry, ML Industry]
Example phrase 2 - "7 years of data engineering experience in SaaS"
Example output 2 - industry: [SaaS]
Example phrase 3 - "Proven leadership experience in fintech or a related industry"
Example output 3 - industry: [fintech, Related]
Step 10 - Read each value of each experience_desc and If any of the values contain technical skills within them or are similar to an existing technical skill requirement then remove the value and only list them under the technical skills section.
Step 11 - Classify each of the experiences that are explicitly described as mandatory, essential, required, or similar as ‘Required’. Classify the experiences that are explicitly described as preferred, nice to have, or similar as ‘Preferred’. If a requirement is not mentioned or not clear, then the experience should be classified as ‘Preferred’. If an experience is only described under job responsibilities or job duties it should be classified as 'Preferred'. Separate these experiences under two objects called ‘Required’ and ‘Preferred’.
Step 12 - If any industries under 'Required' are described as preferred then remove the industry.
Example phrase 1 - "2+ years of experience in Information Technology, preferably in the semiconductor industry"
Example output 1 - industry: []
Step 14 - For each experience, list the minimum years of experience that is explicitly specified for that experience and list 0 if years are not explicitly specified. If a range of years is given, give the smallest year. Label these using a key called ‘minyears’.

Lastly, clean and organize the extracted requirements and responsibilities by following these steps:
Step 1 - If the job description contains the requirements for multiple positions or multiple levels of a position, only keep the 'Required' qualifications that are relevant to the lowest position or most entry level position and then treat the rest of the qualifications as 'Preferred'.
Step 2 - Combine the JSON outputs from skills, education, credentials, and experience into one JSON.
Step 3 - Organize the data under two main objects: 'Required' and 'Preferred', moving the respective skills, education, credentials, and experience under them accordingly.
Step 4 - Output the combined JSON.
"""

In [12]:
def format_reformat_chat_template(row):
    return {
        "role": f"{reformat_prompt}\n\n{row['description']}",
        "content": row["reformatted"]
    }

In [13]:
def format_extract_chat_template(row):
    return {
        "role": f"{extract_prompt}\n\n{row['reformatted']}",
        "content": row["label"]
    }

In [14]:
# test the reformat and extract chat templates
test_row = labeled_df.iloc[0]
print(format_reformat_chat_template(test_row))
print(format_extract_chat_template(test_row))

{'role': '\nYou are a tool used to organize and reformat job descriptions into a structured markdown format \nthat is optimized for information extraction from the ChatGPT 4o model. You must retain all \ninformation from the job description that is related to job requirements, job qualifications, \njob responsibilities, or job duties. You must exclude any information related to descriptions\nof the company, team, salary, travel, location, culture, application process, \nphysical requirements/demands, or benefits. After, output the \nreformatted job description in markdown format.\n\n\nDear Applicant,Hope you are doing well!!!We have an urgent requirement of Senior MES Engineer in Tuscaloosa, AlabamaKindly click to apply if you are available and interested in the job role mentioned belowRole : Senior MES EngineerLocation : Tuscaloosa, AlabamaDuration : Fulltime Hire  Job Description :Responsibility in MES projects: from requirement discussion-> project related document preparation -> IR

In [15]:
# add new columns to hold the reformatting and extraction text fields
labeled_df["reformatted_text"] = labeled_df.apply(format_reformat_chat_template, axis=1)
labeled_df["label_text"] = labeled_df.apply(format_extract_chat_template, axis=1)

In [16]:
labeled_df.head(1)

,description,reformatted,label,reformatted_text,label_text
0,"Dear Applicant,Hope you are doing well!!!We ha...",```markdown\n# Job Title: Senior MES Engineer\...,"{""required"":{""education"":[{""field_of_study"":[""...",{'role': ' You are a tool used to organize and...,{'role': ' You are a tool used to extract the ...


In [17]:
# Create train, validation, test split
train_df, test_df = train_test_split(labeled_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

Train size: 3199
Validation size: 800
Test size: 1000


In [18]:
def combine_reformat_and_extract_examples(dataframe):
  combined_data = []
  for _, row in dataframe.iterrows():
    combined_data.append(row["reformatted_text"])
    combined_data.append(row["label_text"])
  return combined_data


In [19]:
def create_dataset(train_df, val_df, test_df):
    train_data = combine_reformat_and_extract_examples(train_df)
    val_data = combine_reformat_and_extract_examples(val_df)
    test_data = combine_reformat_and_extract_examples(test_df)

    # Convert to Hugging Face DatasetDict
    hf_dataset = DatasetDict({
        "train": Dataset.from_pandas(pd.DataFrame(train_data)),
        "validation": Dataset.from_pandas(pd.DataFrame(val_data)),
        "test": Dataset.from_pandas(pd.DataFrame(test_data))
    })
    return hf_dataset

In [20]:
hf_dataset = create_dataset(train_df, val_df, test_df)

In [21]:
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['role', 'content'],
        num_rows: 6398
    })
    validation: Dataset({
        features: ['role', 'content'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['role', 'content'],
        num_rows: 2000
    })
})


# Tokenize Data

In [8]:
# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained(model_name) # setupchat format in the tutorial
# https://github.com/huggingface/trl/blob/c10cc8995b6fd45f3a876ec98cade97251abe733/trl/models/utils.py#L78

In [9]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [10]:
# just trying to get a feel for the tokenizer
tokenizer_max_length = tokenizer.model_max_length
print(f"Default max length from model configuration: {tokenizer_max_length}")

Default max length from model configuration: 131072


In [25]:
tokenizer.vocab_size

128000

In [26]:
# tokenizer.vocab # This call is not very helpful

In [27]:
tokenizer.all_special_tokens

['<|begin_of_text|>', '<|eot_id|>', '[PAD]']

In [ ]:
# We may want to add the extract and reformat as special tokens, that code is here.
# This takes longer as it may require more training... I'm dropping task tagging for now
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})
# model.resize_token_embeddings(len(tokenizer))

In [28]:
def calculate_token_lengths(dataset):
    input_lengths = [len(tokenizer.encode(entry["role"], add_special_tokens=True)) for entry in dataset]
    output_lengths = [len(tokenizer.encode(entry["content"], add_special_tokens=True)) for entry in dataset]

    max_input_length = max(input_lengths)
    mean_input_length = sum(input_lengths) / len(input_lengths)
    std_input_length = (sum([(length - mean_input_length) ** 2 for length in input_lengths]) / len(input_lengths)) ** 0.5

    max_output_length = max(output_lengths)
    mean_output_length = sum(output_lengths) / len(output_lengths)
    std_output_length = (sum([(length - mean_output_length) ** 2 for length in output_lengths]) / len(output_lengths)) ** 0.5

    # Create DataFrame for descriptive statistics
    input_df = pd.DataFrame({'role_length': input_lengths})
    output_df = pd.DataFrame({'content_length': output_lengths})

    percentile_90_input = np.percentile(input_lengths, 90)
    percentile_90_output = np.percentile(output_lengths, 90)

    # Print descriptive statistics
    print("Role Length Statistics:")
    print(input_df.describe())
    print(percentile_90_input)
    print()
    print("Content Length Statistics:")
    print(output_df.describe())
    print(percentile_90_output)

    return {
        "max_input_length": max_input_length,
        "mean_input_length": mean_input_length,
        "std_input_length": std_input_length,
        "max_output_length": max_output_length,
        "mean_output_length": mean_output_length,
        "std_output_length": std_output_length
    }

In [29]:
length_stats = calculate_token_lengths(hf_dataset["train"])

Role Length Statistics:
       role_length
count  6398.000000
mean   2272.938262
std    1537.854833
min     183.000000
25%     695.250000
50%    3475.000000
75%    3751.000000
max    5781.000000
3906.0

Content Length Statistics:
       content_length
count     6398.000000
mean       318.739294
std        163.671769
min         34.000000
25%        196.000000
50%        291.000000
75%        406.000000
max       1214.000000
536.0


In [11]:
# setting max length to fit our max lengths
max_length = 3906 #90th percentile

# we can also adjust to fit for the distribution of reasonable job description lengths

In [31]:
#  Assuming we have max length set appropriately, we can start here.

def tokenize_function(examples, max_length=max_length):
    # Tokenize the input and output separately
    inputs = tokenizer(
        examples["role"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
    outputs = tokenizer(
        examples["content"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

    # Return tokenized inputs and labels
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": outputs["input_ids"]
    }


In [32]:
tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["role", "content"], # this saves space
    fn_kwargs={'max_length': max_length}
)

Map:   0%|          | 0/6398 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [33]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6398
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})


In [34]:
# save the tokenized dataset
tokenized_dataset.save_to_disk("/content/tokenized_dataset_90th_max_length")

Saving the dataset (0/1 shards):   0%|          | 0/6398 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [35]:
# zip and downlaod to local storage
!zip -r /content/tokenized_dataset_90th_max_length.zip /content/tokenized_dataset_90th_max_length
files.download('/content/tokenized_dataset_90th_max_length.zip')

  adding: content/tokenized_dataset_90th_max_length/ (stored 0%)
  adding: content/tokenized_dataset_90th_max_length/validation/ (stored 0%)
  adding: content/tokenized_dataset_90th_max_length/validation/dataset_info.json (deflated 69%)
  adding: content/tokenized_dataset_90th_max_length/validation/state.json (deflated 38%)
  adding: content/tokenized_dataset_90th_max_length/validation/data-00000-of-00001.arrow (deflated 97%)
  adding: content/tokenized_dataset_90th_max_length/dataset_dict.json (deflated 5%)
  adding: content/tokenized_dataset_90th_max_length/train/ (stored 0%)
  adding: content/tokenized_dataset_90th_max_length/train/dataset_info.json (deflated 69%)
  adding: content/tokenized_dataset_90th_max_length/train/state.json (deflated 38%)
  adding: content/tokenized_dataset_90th_max_length/train/data-00000-of-00001.arrow (deflated 97%)
  adding: content/tokenized_dataset_90th_max_length/test/ (stored 0%)
  adding: content/tokenized_dataset_90th_max_length/test/dataset_info.j

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# LoRa Finetuning

In [ ]:
# load the tokenized dataset
zip_file_path = '/content/tokenized_dataset_90th.zip'
extract_dir = '/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [12]:
tokenized_dataset = DatasetDict.load_from_disk("/content/tokenized_dataset_90th_max_length")

In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True
)

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
    ) # this is set to A100 params

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaFlashAttention2(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=

In [16]:
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(128257, 3072)

In [14]:
#  # setting these to prevent overfitting and runtime
# lora_layers = ["q_proj", "v_proj"] # if we want we can add "k_proj" and "o_proj"

# for name, param in model.named_parameters():
#     if "embed_tokens" in name: # embedding layer
#         param.requires_grad = True

#     elif any(layer in name for layer in lora_layers): # lora layers
#         param.requires_grad = True

#     else:
#         param.requires_grad = False

RuntimeError: only Tensors of floating point and complex dtype can require gradients

In [17]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [ ]:
# trainable_params = [name for name, param in model.named_parameters() if param.requires_grad]
# n_frozen = sum(p.numel() for p in model.parameters() if p.requires_grad)
# n_total = sum(p.numel() for p in model.parameters())
# percent_trainable = n_frozen / n_total

# print(f"Number of frozen parameters: {n_frozen}")
# print(f"Number of total parameters: {n_total}")
# print(f"Percent Non Frozen:  {percent_trainable:.5f}")


In [18]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False, # set this to true when running new examples through
    r=baseline_r,
    lora_alpha=baseline_lora_alpha,
    lora_dropout=baseline_lora_dropout,
    target_modules=modules
)

In [19]:
print(peft_config)

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'gate_proj', 'o_proj', 'down_proj', 'k_proj', 'up_proj', 'v_proj', 'q_proj'}, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


In [20]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,066,752 || trainable%: 0.7511


In [27]:
training_args = TrainingArguments(
    output_dir="/content/3B-Instruct-5k-Lora",
    learning_rate=lr,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    bf16=True,
    # gradient_accumulation_steps=gradient_accumulation_steps,
    # gradient_checkpointing=True,
    report_to="wandb"
)

In [23]:
# for split in ["train", "test"]:
#     max_len = max(len(x) for x in tokenized_dataset[split]["input_ids"])
#     print(f"Max length in {split}: {max_len}")
#     if max_len > max_length:
#         raise ValueError(f"Sequence length in {split} exceeds max_seq_length ({max_length}).")

In [28]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    peft_config=peft_config,
    max_seq_length= max_length,
    tokenizer=tokenizer,
    args=training_args,
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [25]:
print(torch.cuda.memory_summary(device=None, abbreviated=False))


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 2            |        cudaMalloc retries: 4         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  37942 MiB |  38191 MiB | 179197 MiB | 141255 MiB |
|       from large pool |  37736 MiB |  37989 MiB | 178922 MiB | 141186 MiB |
|       from small pool |    205 MiB |    205 MiB |    274 MiB |     69 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  37942 MiB |  38191 MiB | 179197 MiB | 141255 MiB |
|       from large pool |  37736 MiB |  37989 MiB | 178922 MiB |

In [26]:
!nvidia-smi

Wed Nov 27 00:13:30 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   29C    P0              48W / 400W |  40317MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [ ]:
# https://www.datacamp.com/tutorial/fine-tuning-llama-3-2